# ChestX-AI: Deep Learning for Chest X-Ray Diagnosis

[![Python 3.10](https://img.shields.io/badge/Python-3.10-blue.svg)](https://www.python.org/)
[![PyTorch](https://img.shields.io/badge/PyTorch-2.0+-ee4c2c.svg)](https://pytorch.org/)
[![License: MIT](https://img.shields.io/badge/License-MIT-green.svg)](https://opensource.org/licenses/MIT)

Production-grade deep learning system for detecting **14 thoracic diseases** from chest X-ray images.

**Author:** Asfand Yar | **GitHub:** [godpubg345-source/chestx-ai-diagnosis](https://github.com/godpubg345-source/chestx-ai-diagnosis)

## Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import cv2
import warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Model Architecture (DenseNet121)

In [ ]:
class ChestXrayDenseNet(nn.Module):
    DISEASES = ["Atelectasis", "Cardiomegaly", "Effusion", "Infiltration", "Mass",
                "Nodule", "Pneumonia", "Pneumothorax", "Consolidation", "Edema",
                "Emphysema", "Fibrosis", "Pleural_Thickening", "Hernia"]
    
    def __init__(self, num_classes=14, pretrained=True, dropout=0.5):
        super().__init__()
        weights = models.DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None
        self.densenet = models.densenet121(weights=weights)
        num_features = self.densenet.classifier.in_features
        self.densenet.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(num_features, num_classes),
            nn.Sigmoid()
        )
        self.features = self.densenet.features
        self.classifier = self.densenet.classifier
        
    def forward(self, x):
        features = self.features(x)
        out = F.relu(features, inplace=True)
        out = F.adaptive_avg_pool2d(out, (1, 1))
        out = torch.flatten(out, 1)
        return self.classifier(out)

model = ChestXrayDenseNet(pretrained=True).to(device)
model.eval()
print(f"Model: DenseNet121 ({sum(p.numel() for p in model.parameters()):,} params)")

## Grad-CAM (Explainable AI)

In [ ]:
class GradCAM:
    def __init__(self, model):
        self.model = model
        self.model.eval()
        self.gradients = None
        self.activations = None
        self.target_layer = model.features
        
        def forward_hook(module, input, output):
            self.activations = output.detach()
        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0].detach()
        
        self.target_layer.register_forward_hook(forward_hook)
        self.target_layer.register_full_backward_hook(backward_hook)
    
    def generate(self, input_tensor, target_class=None):
        self.model.zero_grad()
        output = self.model(input_tensor)
        if target_class is None:
            target_class = output.argmax(dim=1).item()
        output[0, target_class].backward()
        
        weights = torch.mean(self.gradients[0], dim=(1, 2))
        cam = torch.zeros(self.activations[0].shape[1:], device=self.activations.device)
        for i, w in enumerate(weights):
            cam += w * self.activations[0][i]
        cam = F.relu(cam)
        cam = (cam - cam.min()) / (cam.max() + 1e-8)
        return cam.cpu().numpy(), target_class

gradcam = GradCAM(model)
print("Grad-CAM initialized")

## Inference Pipeline

In [ ]:
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def analyze_xray(image_path):
    image = Image.open(image_path).convert('RGB')
    input_tensor = preprocess(image).unsqueeze(0).to(device).requires_grad_(True)
    
    with torch.no_grad():
        probs = model(input_tensor.clone()).cpu().numpy()[0]
    
    cam, _ = gradcam.generate(input_tensor, probs.argmax())
    cam_resized = cv2.resize(cam, image.size)
    heatmap = cv2.applyColorMap(np.uint8(255 * cam_resized), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    overlay = (0.6 * np.array(image) + 0.4 * heatmap).astype(np.uint8)
    
    return probs, overlay, image

## Demo

In [ ]:
# Create demo image
dummy = np.zeros((512, 512), dtype=np.uint8)
cv2.ellipse(dummy, (256, 280), (180, 200), 0, 0, 360, 80, -1)
cv2.ellipse(dummy, (256, 280), (60, 80), 0, 0, 360, 120, -1)
cv2.imwrite('demo.png', cv2.GaussianBlur(dummy, (15, 15), 0))

probs, overlay, original = analyze_xray('demo.png')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(original, cmap='gray'); axes[0].set_title('X-Ray'); axes[0].axis('off')
axes[1].imshow(overlay); axes[1].set_title('Grad-CAM'); axes[1].axis('off')

sorted_idx = np.argsort(probs)[::-1][:7]
axes[2].barh([ChestXrayDenseNet.DISEASES[i] for i in sorted_idx][::-1], probs[sorted_idx][::-1])
axes[2].set_title('Predictions'); axes[2].set_xlim(0, 1)
plt.tight_layout(); plt.show()

print("\nTop Predictions:")
for i in sorted_idx[:5]:
    print(f"  {ChestXrayDenseNet.DISEASES[i]}: {probs[i]*100:.1f}%")

---
**GitHub:** [godpubg345-source/chestx-ai-diagnosis](https://github.com/godpubg345-source/chestx-ai-diagnosis) | Star if helpful!